# YouTube Transcript RAG System

This notebook is a compact, credential-safe way to exercise the modular backend. The production API and Chrome extension use the same service classes.


In [ ]:
from dotenv import load_dotenv

load_dotenv()  # Reads local .env; never put tokens in this notebook.

from backend.config import get_settings
from backend.service import YoutubeRagService


In [ ]:
settings = get_settings()
settings


## Load a video

Use any valid YouTube watch URL, short link, short, embed, live URL, or raw 11-character video ID. The first call retrieves cached captions or fetches Hindi/English captions, chunks them with timestamps, and upserts them into the shared Pinecone index.


In [ ]:
service = YoutubeRagService(settings)
video_url = "PASTE_A_YOUTUBE_URL_HERE"
video_title = "Optional title"

# Remove the next line after setting video_url.
assert video_url != "PASTE_A_YOUTUBE_URL_HERE", "Set video_url before running this cell."
load_result = await service.load_video(video_url, video_title)
load_result


## Ask grounded questions

Answers are generated only from compressed retrieval context. Sources are structured records, so a UI can render timestamps as seek controls without parsing model output.


In [ ]:
question = "What is the main idea of this video?"
session_id = "notebook-session"
result = await service.chat(load_result["video_id"], question, session_id, video_title)

{"answer": result.answer, "sources": [source.__dict__ for source in result.sources], "route": result.route}


## Evaluate changes

Fill eval/questions.json and eval/answers.json with human-reviewed examples captured from the API, then run python eval/evaluate.py from the project root to calculate RAGAS faithfulness, answer relevancy, context precision, and context recall.
